<a href="https://colab.research.google.com/github/YoungSong99/demonhacks_26/blob/main/3d_main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#
from google.colab import drive
drive.mount("/content/drive")
%cd /content/drive/MyDrive/Colab_Notebooks/YourTomorrow

# install micromamba
!wget -qO micromamba.tar.bz2 https://micro.mamba.pm/api/micromamba/linux-64/latest
!tar -xvjf micromamba.tar.bz2

!export MAMBA_ROOT_PREFIX=/usr/local/micromamba # under here, venv created


# 3D env cannot with yml
!./bin/micromamba create -n 3D_env python=3.10 pip -y

!./bin/micromamba run -n 3D_env pip install --extra-index-url https://download.pytorch.org/whl/cu124 torch==2.4.0 torchvision==0.19.0 # torch first

!./bin/micromamba run -n 3D_env pip install -r 3D_req_py310.txt # requirements.txt

!./bin/micromamba run -n 3D_env pip install xformers==0.0.27.post2 # xformers
!./bin/micromamba run -n 3D_env pip install facenet-pytorch==2.6.0 --no-deps

!git clone https://github.com/weijielyu/FaceLift.git

%cd FaceLift

!../bin/micromamba run -n 3D_env pip install -v --no-build-isolation git+https://github.com/graphdeco-inria/diff-gaussian-rasterization

%cd ..

!pip install fastapi uvicorn nest_asyncio pyngrok

Mounted at /content/drive
[Errno 2] No such file or directory: '/content/drive/MyDrive/Colab_Notebooks/YourTomorrow'
/content
info/files
info/has_prefix
info/index.json
info/test/run_test.sh
info/hash_input.json
info/recipe/C_ARES_LICENSE.txt
info/licenses/C_ARES_LICENSE.txt
info/recipe/conda_build_config.yaml
info/licenses/REPROC_LICENSE.txt
info/recipe/REPROC_LICENSE.txt
info/licenses/NLOHMANN_JSON_LICENSE.txt
info/recipe/NLOHMANN_JSON_LICENSE.txt
info/recipe/CURL_LICENSE.txt
info/licenses/CURL_LICENSE.txt
info/recipe/CPP_FILESYSTEM_LICENSE.txt
info/recipe/ZLIB_LICENSE.txt
info/licenses/ZLIB_LICENSE.txt
info/licenses/LIBNGHTTP2_LICENSE.txt
info/recipe/LIBNGHTTP2_LICENSE.txt
info/paths.json
info/recipe/LIBLZ4_LICENSE.txt
info/licenses/LIBLZ4_LICENSE.txt
info/recipe/SPDLOG_LICENSE.txt
info/licenses/SPDLOG_LICENSE.txt
info/licenses/FMT_LICENSE.txt
info/recipe/FMT_LICENSE.txt
info/licenses/LIBSOLV_LICENSE.txt
info/recipe/LIBSOLV_LICENSE.txt
info/licenses/mamba/LICENSE
info/recipe/build.s

In [ ]:
import asyncio
_colab_semaphore = asyncio.Semaphore(1)

In [ ]:
%%writefile app_3d.py

import asyncio
_colab_semaphore = asyncio.Semaphore(1)

import os, sys
BASE_DIR = os.path.dirname(__file__)
sys.path.insert(0, os.path.join(BASE_DIR, "FaceLift"))

import os, uuid
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.responses import FileResponse, Response, JSONResponse
from three_d_generator import ThreeDGenerator, bundle
from PIL import Image
import io


app = FastAPI()

BASE_DIR = "."
AGING_DIR = os.path.join(BASE_DIR, "aging")
MODEL_DIR = os.path.join(BASE_DIR, "models")
DUMMY_PLY_PATH = os.path.join(BASE_DIR, "gaussians.ply")


@app.post("/api/build-3d")
async def api_build_3d(file: UploadFile = File(...)):
    model_id = str(uuid.uuid4())
    output_dir = os.path.join(MODEL_DIR, model_id)
    os.makedirs(output_dir, exist_ok=True)

    data = await file.read()
    pil_image = Image.open(io.BytesIO(data)).convert("RGB")

    three_d_generator = ThreeDGenerator(pil_image, bundle)
    three_d_generator.generate_3d_img(output_dir=output_dir)

    ply_path = os.path.join(output_dir, "gaussians.ply")

    if not os.path.exists(ply_path):
        return JSONResponse({"error": "PLY generation failed"}, status_code=500)

    return JSONResponse({"model_id": model_id})


@app.get("/api/models/{model_id}")
async def get_model(model_id: str):
    ply_path = os.path.join(MODEL_DIR, model_id, "gaussians.ply")
    if not os.path.exists(ply_path):
        return JSONResponse({"error": "not found"}, status_code=404)

    return FileResponse(
        ply_path,
        media_type="application/octet-stream",
        filename="gaussians.ply"
    )

Overwriting app_3d.py


In [ ]:
!./bin/micromamba run -n3D_env pip install python-multipart

!pip install fastapi uvicorn nest_asyncio pyngrok
!ngrok config add-authtoken
from pyngrok import ngrok
public_url = ngrok.connect(8000)
print(public_url)
!./bin/micromamba run -n 3D_env uvicorn app_3d:app --host 127.0.0.1 --port 8000 --app-dir .

ERROR: Operation cancelled by user


KeyboardInterrupt: 